# AgentOps with Databricks Managed MLflow
**Practical MLflow Workflows for Monitoring, Evaluation & Continuous Improvement**

*Stephanie Alba | Databricks*

## Prerequisites
- Databricks workspace with access to Foundation Model APIs
- MLflow 3.10+ (`mlflow[databricks]>=3.10`)
- Basic familiarity with Python and LLMs

## Learning Objectives
By the end of this notebook you will be able to:
1. **Instrument** an agent with MLflow tracing and structured spans
2. **Evaluate** agent quality using built-in scorers, custom judges, and prompt optimization
3. **Integrate** human feedback signals into your evaluation workflow
4. **Monitor** production quality, latency, and cost metrics
5. **Govern** agent lifecycle with evaluation gates and run comparisons

> **Workshop flow:** The companion slide deck frames the *why*; this notebook is the *how*.

In [0]:
%pip install "mlflow[databricks]>=3.13.0" openai "gepa>=0.0.26" litellm "anyio<4" --quiet
dbutils.library.restartPython()

In [0]:
%skip
from mlflow import MlflowClient
client = MlflowClient()
prompt_name = "mfg_mc_se_sa.agent_ops.support_agent_prompt"
search_response = client.search_prompt_versions(prompt_name)  # search_response is iterable, no need to access prompt_versions attribute
for version in search_response:
    client.delete_prompt_version(prompt_name, str(version.version))
client.delete_prompt(prompt_name)

In [0]:
# %sql
# drop schema mfg_mc_se_sa.agent_ops cascade;
# create schema mfg_mc_se_sa.agent_ops;

## Why Agent Observability Matters

### Core AgentOps Challenges

Building production agents is fundamentally harder than building traditional ML models. Here are the six challenges every team faces:

| # | Challenge | Description |
|---|-----------|-------------|
| 1 | **Leveraging Enterprise Context** | Agents must ground responses in proprietary data -- knowledge bases, internal docs, databases -- without leaking or hallucinating. |
| 2 | **Reliability & Consistency** | Non-deterministic outputs mean the same question can yield different answers. You need guardrails to ensure consistent quality. |
| 3 | **End-to-End Observability** | Multi-step chains (retrieval -> reasoning -> tool use -> generation) create opaque pipelines. You need trace-level visibility. |
| 4 | **Safety & Guardrails** | Agents can generate harmful, biased, or off-topic content. Automated safety checks must run on every invocation. |
| 5 | **Scalability** | What works for 10 queries/day breaks at 10,000. Latency, cost, and quality all degrade differently under load. |
| 6 | **Maintainability** | Models change, data drifts, user expectations evolve. Without versioning and eval gates, regressions ship silently. |

> These challenges are why "just log to a database" isn't enough. You need a structured observability framework.

### MLflow Core Concepts for Agent Tracking
*The building blocks you'll use throughout this workshop*

| Concept | What It Is | Example |
|---------|------------|---------|
| **Experiment** | A named container for related runs | `"customer-support-agent-v2"` |
| **Run** | One execution instance capturing params, metrics, artifacts & traces | A single evaluation pass over 50 test questions |
| **Metric** | Numeric signal logged over time | `latency_ms`, `quality_score`, `token_count` |
| **Artifact** | Any file: traces, eval results, prompt templates, datasets | `eval_results.json`, `trace.json` |
| **Trace** | A structured record of an agent's execution with nested spans | Retrieval span -> Generation span -> Tool span |
| **Tag** | Key-value metadata for organizing and filtering runs | `{"env": "prod", "version": "1.2.0"}` |
| **Prompt** | A versioned prompt template stored in the Prompt Registry | `mlflow.genai.register_prompt("support_agent", template=...)` |

> An **Experiment** contains many **Runs**. Each **Run** captures **Metrics**, **Artifacts**, and **Traces**. **Prompts** are versioned independently and can be optimized over time.

## Pillar 1: Instrument and Trace an Agent
---

In this section we'll build a simple RAG-style support agent, instrument it with MLflow tracing, and explore the trace UI.

### Unity Catalog-Backed Traces

For production use, you can store traces in **Unity Catalog** Delta tables. See the [migration guide](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc) for setup details.

| Benefit | Description |
|---------|-------------|
| **SQL queryability** | Query traces with SQL — join with other Delta tables, build dashboards |
| **Governance** | UC permissions, column masking, audit logs apply to trace data |
| **Durability** | Traces stored as Delta tables with full ACID guarantees |
| **Cross-workspace access** | Share trace data across workspaces via Unity Catalog |

In [0]:
import mlflow
import os
from mlflow.entities.trace_location import UnityCatalog

# ── Configure MLflow experiment with UC-backed traces ─────────
mlflow.set_tracking_uri("databricks")
os.environ["MLFLOW_TRACING_SQL_WAREHOUSE_ID"] = "bce0a02b2be86f1b"

UC_CATALOG = "mfg_mc_se_sa"
UC_SCHEMA = "agent_ops"

# Derive experiment name from current notebook path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
experiment_name = f"{notebook_path} Experiment"

experiment = mlflow.set_experiment(
    experiment_name=experiment_name,
    trace_location=UnityCatalog(
        catalog_name=UC_CATALOG,
        schema_name=UC_SCHEMA,
        table_prefix="workshop",
    ),
)

mlflow.openai.autolog()

# Suppress noisy MLflow warnings
import logging
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

import warnings
warnings.filterwarnings("ignore", message=".*Inferred schema contains integer column.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*default dtype for empty Series.*")

print("MLflow tracking configured!")
print(f"  Notebook: {notebook_path}")
print(f"  Experiment: {experiment.name}")
print(f"  Traces stored in UC: {UC_CATALOG}.{UC_SCHEMA}")

In [0]:
import mlflow.genai

# ── Register prompt in the Prompt Registry ────────────────
# Prompts are UC-backed on Databricks — use catalog.schema.name format
support_prompt = mlflow.genai.register_prompt(
    name="mfg_mc_se_sa.agent_ops.support_agent_prompt",
    template=(
        "You are a customer support agent. Answer the question based on "
        "your general knowledge. Do not reference any specific policies or details. "
        "Answer in one sentence or less. Do not explain. "
        "Do not provide details like timeframes, prices, or conditions.\n\n"
        "Context:\n{{context}}\n\n"
        "Question: {{question}}"
    ),
)

print(f"Prompt registered: {support_prompt.name}")
print(f"  Version: {support_prompt.version}")
print(f"  URI: {support_prompt.uri}")

### Define a Knowledge Base
We'll use a simple in-memory knowledge base to simulate retrieval. In production, this would be a vector store, search index, or database.

In [0]:
from openai import OpenAI

# ── Simple knowledge base ─────────────────────────────────
KNOWLEDGE_BASE = {
    "returns": "Our return policy allows returns within 30 days of purchase. Items must be in original condition with receipt. Refunds are processed within 5-7 business days to the original payment method.",
    "shipping": "Standard shipping takes 5-7 business days. Express shipping (2-day) is available for $12.99. Free shipping on orders over $50. International shipping available to 40+ countries.",
    "warranty": "All electronics come with a 2-year manufacturer warranty. Extended warranty available for purchase. Warranty covers defects in materials and workmanship, not accidental damage.",
    "pricing": "We offer price matching within 14 days of purchase. Show us a competitor's lower price and we'll match it. Excludes marketplace sellers and clearance items.",
    "account": "You can manage your account at account.example.com. Reset passwords via email verification. Two-factor authentication is available and recommended for all accounts.",
}

@mlflow.trace(name="retrieve_context", span_type="RETRIEVER")
def retrieve_context(query: str) -> str:
    """Keyword-based retrieval from knowledge base."""
    query_lower = query.lower()
    results = []
    for topic, content in KNOWLEDGE_BASE.items():
        if topic in query_lower or any(word in query_lower for word in topic.split()):
            results.append(content)
    # Default fallback
    if not results:
        results = [KNOWLEDGE_BASE["returns"]]
    return "\n\n".join(results)

print("Knowledge base loaded with topics:", list(KNOWLEDGE_BASE.keys()))

In [0]:
import os

@mlflow.trace
def run_support_agent(question: str) -> dict:
    """RAG support agent with traced retrieval and generation spans."""
    # Step 1: Retrieve context
    context = retrieve_context(question)

    # Step 2: Load the versioned prompt from the registry
    prompt = mlflow.genai.load_prompt(support_prompt.uri)
    system_message = prompt.format(question=question, context=context)

    # Step 3: Generate response using Databricks Foundation Model
    client = OpenAI(
        api_key=os.environ.get("DATABRICKS_TOKEN", dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()),
        base_url=f"{mlflow.utils.databricks_utils.get_workspace_url()}/serving-endpoints"
    )

    with mlflow.start_span(name="llm_generation", span_type="LLM") as span:
        span.set_inputs({"question": question, "context": context})

        response = client.chat.completions.create(
            model="databricks-claude-sonnet-4",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": question}
            ],
            max_tokens=300,
            temperature=0.1
        )

        answer = response.choices[0].message.content
        span.set_outputs({"answer": answer})

    return {"response": answer}

print("Support agent defined and ready!")

In [0]:
# ── Run the agent on test questions ─────────────────────────
test_questions = [
    "What is your return policy?",
    "How long does shipping take?",
    "Does my laptop come with a warranty?",
    "Can you match a competitor's price?",
    "How do I reset my password?",
]

print("Running agent on 5 test questions...\n")
for i, question in enumerate(test_questions, 1):
    result = run_support_agent(question)
    print(f"Q{i}: {question}")
    print(f"A{i}: {result['response'][:150]}...")
    print()

print("All traces logged to MLflow! Check the Traces tab in the experiment UI.")

### Checkpoint: Explore the Trace UI

Now that we've run the agent, let's explore what MLflow captured.

1. Click **Experiments** in the left sidebar
2. Open the experiment matching your notebook name
3. Click the **Traces** tab
4. Click on any trace to see the waterfall view:
   - **support_agent** (root span)
     - **retrieve_context** (retriever span) -- inputs, outputs, latency
     - **llm_generation** (LLM span) -- prompt, response, token counts

> **Key insight:** Every span captures inputs, outputs, and timing. This is how you debug multi-step agent failures in production.

## Pillar 2: Evaluation
---

Tracing tells you *what happened*. Evaluation tells you *how well it went*. This 6-step loop drives continuous improvement:

| Step | Action | Details |
|------|--------|---------|
| 1. **Define** | Identify quality signals | What matters? Correctness, helpfulness, safety, groundedness |
| 2. **Collect** | Curate a golden dataset | Input/output pairs representative of production traffic |
| 3. **Evaluate** | Run `mlflow.genai.evaluate()` | Scorers produce Feedback on every trace |
| 4. **Analyze** | Compare runs in MLflow UI | Identify regressions, improvements, and edge cases |
| 5. **Improve** | Adjust prompts, retrieval, or model | Re-run evaluation. Ship only when metrics improve. |
| 6. **Repeat** | Add failing cases to golden dataset | The loop compounds quality over time |

> **Start small:** Even 50 well-curated examples are transformative. You don't need thousands to begin.

In [0]:
# ── Prepare evaluation dataset ──────────────────────────────
eval_data = [
    {
        "inputs": {"question": "What is your return policy?"},
        "expectations": {"expected_response": "Returns are allowed within 30 days of purchase. Items must be in original condition with receipt. Refunds processed in 5-7 business days."},
    },
    {
        "inputs": {"question": "How long does standard shipping take?"},
        "expectations": {"expected_response": "Standard shipping takes 5-7 business days. Express 2-day shipping is $12.99. Free shipping on orders over $50."},
    },
    {
        "inputs": {"question": "What does the warranty cover?"},
        "expectations": {"expected_response": "Electronics have a 2-year manufacturer warranty covering defects in materials and workmanship. Extended warranty is available. Accidental damage is not covered."},
    },
    {
        "inputs": {"question": "Do you offer price matching?"},
        "expectations": {"expected_response": "Price matching is available within 14 days of purchase for competitor prices, excluding marketplace sellers and clearance items."},
    },
    {
        "inputs": {"question": "How do I set up two-factor authentication?"},
        "expectations": {"expected_response": "Manage your account at account.example.com. Two-factor authentication is available and recommended for all accounts."},
    },
]

print(f"Evaluation dataset: {len(eval_data)} examples")
for i, row in enumerate(eval_data, 1):
    print(f"  {i}. {row['inputs']['question']}")

In [0]:
from mlflow.genai.scorers import Correctness, RelevanceToQuery, Safety, Guidelines

# Suppress internal MLflow/pandas warnings we can't fix from user code
import warnings
warnings.filterwarnings("ignore", message=".*Inferred schema contains integer column.*")
warnings.filterwarnings("ignore", category=FutureWarning, message=".*default dtype for empty Series.*")

# ── Run automated evaluation ────────────────────────────────
results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=run_support_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Safety(),
    ],
)

print("Evaluation complete! Check the MLflow Traces tab for per-row results.")
print(f"  Run ID: {results.run_id}")

###Prompt Optimization
---

We registered our prompt in the Prompt Registry at the start of this notebook. Now we can systematically improve it using evaluation data and LLM-driven reflection:

1. **Define** a predict function that loads the registered prompt
2. **Run** `mlflow.genai.optimize_prompts()` with training data and scorers
3. **Get back** an improved prompt version, automatically registered as a new version

| Optimizer | How It Works | Best For |
|-----------|-------------|----------|
| **GepaPromptOptimizer** | Genetic-Pareto optimization with natural language reflection. Up to 35x fewer iterations than brute force. | High-stakes tasks with substantial eval datasets |
| **MetaPromptOptimizer** | Zero-shot or few-shot metaprompting. Fast, low cost. | Quick improvements, small datasets or no data at all |

> **Docs:** [mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts](https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/)

In [0]:
from openai import OpenAI

# ── Step 1: Define a predict function that loads the registered prompt ──
def predict_with_prompt(question: str, context: str) -> str:
    """Predict function that loads the prompt from the registry."""
    prompt = mlflow.genai.load_prompt(f"prompts:/{support_prompt.name}/{support_prompt.version}")

    client = OpenAI(
        api_key=os.environ.get("DATABRICKS_TOKEN", dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()),
        base_url=f"{mlflow.utils.databricks_utils.get_workspace_url()}/serving-endpoints"
    )
    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=[{"role": "user", "content": prompt.format(question=question, context=context)}],
        max_tokens=300,
        temperature=0.1,
    )
    return response.choices[0].message.content

# Quick test
test_result = predict_with_prompt("What is your return policy?", KNOWLEDGE_BASE["returns"])
print(f"Test response: {test_result[:150]}...")

In [0]:
from mlflow.genai.optimize import GepaPromptOptimizer
import litellm

# ── Step 2: Prepare training data for optimization ──────────
train_data = [
    {
        "inputs": {"question": "What is your return policy?", "context": KNOWLEDGE_BASE["returns"]},
        "expectations": {"expected_response": "Returns within 30 days with receipt. Refunds in 5-7 business days."},
    },
    {
        "inputs": {"question": "How long does shipping take?", "context": KNOWLEDGE_BASE["shipping"]},
        "expectations": {"expected_response": "Standard shipping 5-7 business days. Express 2-day for $12.99. Free over $50."},
    },
    {
        "inputs": {"question": "What does the warranty cover?", "context": KNOWLEDGE_BASE["warranty"]},
        "expectations": {"expected_response": "2-year warranty on electronics for defects. Extended warranty available. No accidental damage."},
    },
    {
        "inputs": {"question": "Do you price match?", "context": KNOWLEDGE_BASE["pricing"]},
        "expectations": {"expected_response": "Price matching within 14 days, excluding marketplace sellers and clearance."},
    },
    {
        "inputs": {"question": "How do I enable 2FA?", "context": KNOWLEDGE_BASE["account"]},
        "expectations": {"expected_response": "Go to account.example.com. Two-factor authentication is available for all accounts."},
    },
]

# ── Step 3: Run prompt optimization ─────────────────────────
optimizer = GepaPromptOptimizer(
    reflection_model="databricks:/databricks-claude-sonnet-4",
    max_metric_calls=50,  # Keep cost low for workshop demo
)

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_with_prompt,
    train_data=train_data,
    prompt_uris=[support_prompt.uri],
    optimizer=optimizer,
    scorers=[Correctness(model="endpoints:/databricks-claude-sonnet-4"),
             RelevanceToQuery(model="endpoints:/databricks-claude-sonnet-4")],
)

print("\n=== Prompt Optimization Results ===")
print(f"  Initial score: {result.initial_eval_score}")
print(f"  Final score:   {result.final_eval_score}")
for prompt in result.optimized_prompts:
    print(f"\n  Optimized prompt '{prompt.name}' v{prompt.version}:")
    print(f"  {prompt.template[:200]}...")

### Scorers Deep Dive
*Built-in scorers vs. custom judges*

MLflow 3 provides a unified **scorer** interface. Built-in scorers cover common quality dimensions; custom judges and code-based scorers handle domain-specific needs:

| Scenario | Approach | Example |
|----------|----------|---------|
| Factual correctness | `Correctness()` | Compares output to expected response |
| Relevant to the user's query | `RelevanceToQuery()` | Checks output addresses the question |
| Grounded in retrieved context | `RetrievalGroundedness()` | Verifies output uses retrieval results |
| Safety / harmful content | `Safety()` | Detects unsafe or harmful outputs |
| Custom natural-language rules | `Guidelines(name=..., guidelines=...)` | Pass/fail against your own rules |
| Fully custom LLM judge | `make_judge(name=..., instructions=...)` | Numerical, categorical, or boolean scores |
| Deterministic / code-based | `@scorer` decorator | Exact match, format validation, latency checks |

> **Key principle:** Scorers receive a **Trace** and return **Feedback** attached to that trace. The same scorers work in both development evaluation and production monitoring.

In [0]:
from mlflow.genai.scorers import Guidelines, scorer
from mlflow.entities import Feedback

# ── Define custom scorers ─────────────────────────────────────

# Option 1: Guidelines scorer — pass/fail against natural-language rules
helpfulness_judge = Guidelines(
    name="helpfulness",
    guidelines=[
        "The response must directly address the user's question",
        "The response must provide specific, actionable information",
        "The response must not redirect the user without answering",
    ],
)

# Option 2: Code-based scorer — deterministic checks using @scorer
@scorer
def response_length_check(outputs: str) -> Feedback:
    """Check that the response is substantive (not too short or too long)."""
    word_count = len(outputs.split()) if outputs else 0
    if word_count < 10:
        return Feedback(value="no", rationale=f"Response too short ({word_count} words). Expected at least 10.")
    elif word_count > 500:
        return Feedback(value="no", rationale=f"Response too long ({word_count} words). Expected under 500.")
    else:
        return Feedback(value="yes", rationale=f"Response length is appropriate ({word_count} words).")

print("Custom scorers defined:")
print("  1. helpfulness (Guidelines judge — pass/fail against 3 rules)")
print("  2. response_length_check (code-based scorer — deterministic)")

In [0]:
# ── Run evaluation with custom scorers ─────────────────────
custom_results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=run_support_agent,
    scorers=[
        Correctness(),
        helpfulness_judge,
        response_length_check,
    ],
)

print("Custom evaluation complete! Check the MLflow Traces tab for per-row feedback.")
print(f"  Run ID: {custom_results.run_id}")

### Multi-Turn Conversation Evaluation

Single-turn evaluation scores individual question-answer pairs. But real agents handle **multi-turn conversations** where quality emerges over several exchanges — follow-up questions, clarifications, and context that builds across turns.

MLflow provides two approaches:

| Approach | How It Works | Best For |
|----------|-------------|----------|
| **Evaluate existing conversations** | Pass traced sessions to `mlflow.genai.evaluate()` | Production conversations you've already captured |
| **Simulate conversations** | `ConversationSimulator` generates multi-turn dialogues with a simulated user | Testing new agents before deployment |

**Built-in multi-turn scorers:**
- `ConversationCompleteness()` — Did the agent fully resolve the user's goal?
- `UserFrustration()` — Did the user become frustrated? Did the agent make it worse?

In [0]:
from mlflow.genai.simulators import ConversationSimulator
from mlflow.genai.scorers import ConversationCompleteness, UserFrustration

# ── Ensure DATABRICKS_HOST is set for the simulator's internal LLM calls ──
workspace_url = mlflow.utils.databricks_utils.get_workspace_url()
if not workspace_url.startswith("http"):
    workspace_url = f"https://{workspace_url}"
os.environ["DATABRICKS_HOST"] = workspace_url
if "DATABRICKS_TOKEN" not in os.environ:
    os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# ── Define a multi-turn predict function ──────────────────────
# ConversationSimulator passes chat messages (list of dicts) to predict_fn
def multi_turn_predict(input: list[dict], **kwargs) -> str:
    """Agent that handles multi-turn conversations."""
    all_context = "\n\n".join(KNOWLEDGE_BASE.values())
    messages = [
        {"role": "system", "content": f"You are a helpful customer support agent. Use the following context to answer questions. If the context doesn't contain the answer, say so.\n\n{all_context}"}
    ] + input

    client = OpenAI(
        api_key=os.environ["DATABRICKS_TOKEN"],
        base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints"
    )
    response = client.chat.completions.create(
        model="databricks-claude-sonnet-4",
        messages=messages,
        max_tokens=300,
        temperature=0.1,
    )
    return response.choices[0].message.content

print("Multi-turn predict function defined!")

In [0]:
# ── Simulate and evaluate multi-turn conversations ────────
simulator = ConversationSimulator(
    test_cases=[
        {
            "goal": "Get a full understanding of the return policy including timeframes and conditions",
            "persona": "You are a new customer with a straightforward question",
        },
        {
            "goal": "Compare shipping options and decide which is best for an urgent order",
            "persona": "You are a busy professional who needs a quick, clear answer",
        },
        {
            "goal": "Understand warranty coverage and whether accidental damage is included",
            "persona": "You are a skeptical customer who asks detailed follow-up questions",
        },
    ],
    max_turns=4,
    user_model="databricks:/databricks-claude-sonnet-4",
)

multi_turn_results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=multi_turn_predict,
    scorers=[ConversationCompleteness(), UserFrustration()],
)

print("Multi-turn evaluation complete!")
print(f"  Run ID: {multi_turn_results.run_id}")
print("\nCheck the MLflow Traces tab — each simulated conversation appears as a session.")

## Pillar 3: Human Feedback
---
*Align automated scores with real-world quality signals*

Automated evaluation is necessary but not sufficient. Human feedback grounds your metrics in reality.

| Signal Type | Source | How to Capture |
|-------------|--------|----------------|
| **Explicit** | User clicks thumbs up/down | `mlflow.log_feedback(trace_id, name="user_feedback", value=True/False)` |
| **Corrections** | User rephrases or edits response | `mlflow.log_feedback()` with rationale capturing the correction |
| **Annotations** | Internal reviewers label quality | `mlflow.log_feedback()` with `AssessmentSourceType.HUMAN` |
| **Implicit** | User follows up, asks for clarification, or abandons | Infer satisfaction from session behavior |

**How it works in production:**
1. Your agent returns the `trace_id` with each response to the frontend
2. When a user clicks thumbs up/down, your frontend calls your backend API
3. Your backend calls `mlflow.log_feedback()` with the `trace_id` and the user's rating
4. Both automated assessments and human feedback appear side by side on each trace

**Ready to scale?** Use [Labeling Sessions](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/concepts/labeling-sessions) to coordinate structured feedback collection across your team.

In [0]:
import random
import numpy as np
from mlflow.entities.assessment import AssessmentSource, AssessmentSourceType

# ── Human Feedback: Log user ratings on real traces ───────────
# In production, your app returns the trace_id with each response.
# When a user clicks thumbs up/down, your backend calls mlflow.log_feedback().

eval_traces = mlflow.search_traces(run_id=results.run_id)
print(f"Logging human feedback on {len(eval_traces)} evaluation traces...\n")

# Simulate end-user feedback on each trace
# In production, this comes from your UI — thumbs up/down, star ratings, corrections
reviewers = ["reviewer_alice", "reviewer_bob", "reviewer_carol"]

for _, trace in eval_traces.iterrows():
    trace_id = trace["trace_id"]
    is_satisfied = random.random() < 0.80  # Simulate ~80% user satisfaction

    mlflow.log_feedback(
        trace_id=trace_id,
        name="user_feedback",
        value=is_satisfied,
        rationale="Response was helpful and accurate" if is_satisfied
            else "Response was missing key details or was unhelpful",
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id=random.choice(reviewers),
        ),
    )

print("Human feedback logged on all evaluation traces!")
print("Open any trace in the MLflow UI — you'll see both automated")
print("assessments AND human feedback side by side.\n")

# ── Compare automated judges vs human feedback ────────────────
# Re-fetch traces to include the newly logged human feedback
traces_with_feedback = mlflow.search_traces(run_id=results.run_id)

agreed, total = 0, 0
for _, trace in traces_with_feedback.iterrows():
    assessments = trace.get("assessments", [])
    human_fb = None
    auto_pass = True

    for a in assessments:
        source = a.source if hasattr(a, "source") else a.get("source")
        value = a.value if hasattr(a, "value") else a.get("value")
        source_type = None
        if source:
            source_type = source.source_type if hasattr(source, "source_type") else (source.get("source_type") if isinstance(source, dict) else None)
        if source_type == "HUMAN":
            human_fb = value
        elif value in ("no", "fail", False):
            auto_pass = False

    if human_fb is not None:
        total += 1
        if human_fb == auto_pass:
            agreed += 1

print("=" * 55)
print("  HUMAN-JUDGE ALIGNMENT")
print("=" * 55)
print(f"  Traces with human feedback:  {total}")
if total > 0:
    print(f"  Human-judge agreement:       {agreed}/{total} ({agreed/total:.0%})")
    print(f"  Disagreements to review:     {total - agreed}")
    print(f"\n  Disagreements are the most valuable data points —")
    print(f"  use them to refine your scorer prompts.")
print("=" * 55)

## Pillar 4: Production Monitoring
*Track quality, latency, and cost in production*

Development evaluation catches regressions before deployment. Production monitoring catches issues that only appear under real-world conditions.

| Capability | What to Track | Databricks Tool |
|------------|---------------|-----------------|
| **Quality dashboards** | Automated eval scores aggregated hourly/daily | Lakehouse Monitoring + MLflow metrics |
| **Trace sampling** | Random sample of production traces for manual review | MLflow Traces + scheduled jobs |
| **Latency monitoring** | p50/p95/p99 latency per span, overall request latency | Lakehouse Monitoring |
| **Cost tracking** | Token usage, model calls, and $ cost per request | MLflow metrics + billing API |
| **Drift detection** | Input distribution shifts, topic drift, quality degradation | Lakehouse Monitoring drift profiles |
| **Alerting** | Threshold-based or anomaly-based alerts on any metric | Databricks Alerts + SQL Alerts |


In [0]:
from datetime import datetime

# ── Production Monitoring: Analyze Real Traces ────────────────
# In production you'd run this on a schedule (e.g., hourly Databricks Job).
# Here we analyze the traces we've already generated in this workshop.

traces_df = mlflow.search_traces(
    locations=[experiment.experiment_id],
)
print(f"Analyzing {len(traces_df)} traces from experiment\n")

# ── Extract metrics from real trace data ──────────────────────
latencies_ms = traces_df["execution_duration"].dropna().values  # already in ms
error_count = (traces_df["state"] == "ERROR").sum()
total = len(traces_df)

# ── Compute percentiles and log as a monitoring run ───────────
with mlflow.start_run(run_name="prod-monitoring-report"):
    metrics = {
        "monitor/trace_count": total,
        "monitor/error_rate": error_count / total if total else 0,
        "monitor/latency_p50_ms": float(np.percentile(latencies_ms, 50)),
        "monitor/latency_p95_ms": float(np.percentile(latencies_ms, 95)),
        "monitor/latency_p99_ms": float(np.percentile(latencies_ms, 99)),
        "monitor/latency_max_ms": float(np.max(latencies_ms)),
    }
    mlflow.log_metrics(metrics)
    mlflow.set_tags({
        "monitoring.type": "production_report",
        "monitoring.timestamp": datetime.now().isoformat(),
        "env": "production",
    })

    # ── Monitoring report ─────────────────────────────────────
    print("=" * 50)
    print("  PRODUCTION MONITORING REPORT")
    print("=" * 50)
    print(f"  Traces analyzed:  {total}")
    print(f"  Errors:           {error_count} ({metrics['monitor/error_rate']:.1%})")
    print(f"\n  Latency Distribution (ms):")
    print(f"    p50:  {metrics['monitor/latency_p50_ms']:,.0f}")
    print(f"    p95:  {metrics['monitor/latency_p95_ms']:,.0f}")
    print(f"    p99:  {metrics['monitor/latency_p99_ms']:,.0f}")
    print(f"    max:  {metrics['monitor/latency_max_ms']:,.0f}")

    # ── Threshold-based alerts ────────────────────────────────
    LATENCY_SLA = 10_000  # 10s SLA
    ERROR_THRESHOLD = 0.05  # 5% error budget

    alerts = []
    if metrics["monitor/latency_p95_ms"] > LATENCY_SLA:
        alerts.append(f"p95 latency ({metrics['monitor/latency_p95_ms']:,.0f}ms) exceeds {LATENCY_SLA:,}ms SLA")
    if metrics["monitor/error_rate"] > ERROR_THRESHOLD:
        alerts.append(f"Error rate ({metrics['monitor/error_rate']:.1%}) exceeds {ERROR_THRESHOLD:.0%} threshold")

    print(f"\n  Alerts ({len(alerts)}):")
    if alerts:
        for alert in alerts:
            print(f"    [!] {alert}")
        mlflow.set_tag("monitoring.alert_triggered", "true")
    else:
        print("    All metrics within SLA thresholds")
        mlflow.set_tag("monitoring.alert_triggered", "false")
    print("=" * 50)

## Pillar 5: Governance & Lifecycle
*Ship responsibly at enterprise scale*

| Practice | Description | Databricks Tool |
|----------|-------------|-----------------|
| **Model Registry** | Version every model, prompt template, and retrieval config | Unity Catalog Model Registry |
| **Eval gates in CI/CD** | Run `mlflow.evaluate()` on every PR that touches agent behavior | Databricks Jobs + GitHub Actions |
| **Deployment stages** | Promote through dev -> staging -> prod with full lineage | UC model aliases + deployment tags |
| **Audit trails** | Record who changed what, when, and why | Unity Catalog lineage + MLflow tags |
| **Access control** | Team-level experiment permissions, RBAC on model endpoints | Workspace permissions + UC grants |
| **Data governance** | Govern traces containing sensitive information | UC-backed traces + column masking |

**CI/CD Evaluation Gate (pseudocode):**
```python
# In your CI/CD pipeline:
from mlflow.genai.scorers import Correctness, Safety, Guidelines

results = mlflow.genai.evaluate(
    data=golden_dataset,
    predict_fn=new_agent,
    scorers=[Correctness(), Safety(), Guidelines(name="quality", guidelines=...)],
)
traces = mlflow.search_traces(run_id=results.run_id)
# Check assessments for failures and block deployment if quality regresses
```

In [0]:
  # ── CI/CD Eval Gate: Enforce quality before deployment ────                                                                                     
  eval_traces = mlflow.search_traces(run_id=results.run_id)                                                                                        
                                                            
  if len(eval_traces) > 0:
      scorer_results = {}
      for _, trace in eval_traces.iterrows():
          assessments = trace.get("assessments", [])
          if assessments:
              for assessment in assessments:
                  name = assessment.name if hasattr(assessment, "name") else assessment.get("name") if hasattr(assessment, "get") else None
                  value = assessment.value if hasattr(assessment, "value") else assessment.get("value") if hasattr(assessment, "get") else None
                  source = assessment.source if hasattr(assessment, "source") else assessment.get("source") if hasattr(assessment, "get") else None

                  if name is None:
                      continue
                  source_type = None
                  if source:
                      source_type = source.source_type if hasattr(source, "source_type") else (source.get("source_type") if isinstance(source,
  dict) else None)
                  if source_type == "HUMAN":
                      continue

                  if name not in scorer_results:
                      scorer_results[name] = {"pass": 0, "fail": 0}
                  if value in ("yes", "pass", True):
                      scorer_results[name]["pass"] += 1
                  else:
                      scorer_results[name]["fail"] += 1

      QUALITY_GATE = 0.80
      gate_passed = True

      print("=" * 55)
      print("  CI/CD EVAL GATE REPORT")
      print("=" * 55)
      print(f"  Traces evaluated: {len(eval_traces)}")
      print(f"  Quality gate:     {QUALITY_GATE:.0%} pass rate per scorer\n")
      print(f"  {'Scorer':<30} {'Pass Rate':>10} {'Result':>8}")
      print(f"  {'-'*30} {'-'*10} {'-'*8}")

      for scorer_name, counts in sorted(scorer_results.items()):
          total_count = counts["pass"] + counts["fail"]
          rate = counts["pass"] / total_count if total_count else 0
          status = "PASS" if rate >= QUALITY_GATE else "FAIL"
          if status == "FAIL":
              gate_passed = False
          print(f"  {str(scorer_name):<30} {rate:>9.0%} {'  ' + status:>8}")

      print(f"\n  {'='*48}")
      if gate_passed:
          print("  DEPLOYMENT DECISION: APPROVED")
          print("  All scorers meet the quality threshold.")
      else:
          print("  DEPLOYMENT DECISION: BLOCKED")
          print("  One or more scorers below threshold. Investigate")
          print("  failing traces before promoting to production.")
      print("=" * 55)
  else:
      print("No evaluation traces found. Run the evaluation cells above first!")

## Practical Tips for Production Deployments
---
*Lessons from teams who've done this at scale*

| # | Tip | Why It Matters |
|---|-----|---------------|
| 1 | **Start with a golden dataset of 50-100 examples** | Even a small, well-curated eval set is transformative. Prioritize coverage of known failure modes and edge cases. |
| 2 | **Log costs alongside quality** | Track `token_count` and `model_id` alongside `quality_score`. Cost/quality tradeoffs are your most important business metric. |
| 3 | **Tag runs with environment and version** | Use `tags: {"env": "prod", "version": "1.2.0", "git_sha": "abc123"}`. This makes debugging regressions 10x faster. |
| 4 | **Validate your LLM judge before trusting it** | Run 50 examples through both your judge and human raters. If correlation < 0.7, revisit your `grading_prompt`. |
| 5 | **Sample production traces for manual review** | Don't rely solely on automated scores. Review 10-20 traces weekly to catch issues metrics miss. |
| 6 | **Version your eval datasets** | Log eval datasets as MLflow artifacts. When you add new examples, you can compare performance across dataset versions. |

## Summary
---

| Pillar | What We Covered | Key MLflow API |
|--------|----------------|----------------|
| **1. Tracing & Observability** | Instrumented an agent with nested spans for retrieval and generation | `@mlflow.trace`, `mlflow.start_span()` |
| **2. Evaluation** | Ran automated evaluation with built-in scorers, custom judges, and prompt optimization | `mlflow.genai.evaluate()`, `Guidelines`, `@scorer`, `mlflow.genai.optimize_prompts()` |
| **3. Human Feedback** | Simulated human feedback collection and computed judge agreement rates | `mlflow.log_metric()`, correlation analysis |
| **4. Production Monitoring** | Logged production-style metrics (latency, quality, cost) at scale | `mlflow.log_metrics()`, tags, namespaced metrics |
| **5. Governance & Lifecycle** | Reviewed trace assessments and simulated CI/CD eval gates | `mlflow.search_traces()`, assessments |

### Next Steps
- **Expand your golden dataset** with real production examples
- **Set up Lakehouse Monitoring** for continuous quality tracking
- **Integrate eval gates** into your CI/CD pipeline
- **Explore the OSS notebook** if you work with self-hosted MLflow
- **Watch for MLflow Updates!** There are innovations coming all the time!

<img src="./Screenshot 2026-06-02 at 4.33.49 PM.png" width="400">

### Resources
- [MLflow Tracing Docs](https://mlflow.org/docs/latest/tracing.html)
- [MLflow GenAI Evaluation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)
- [MLflow Prompt Optimization](https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/)
- [Migrate Traces to Unity Catalog](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc)
- [Databricks MLflow 3 GenAI](https://docs.databricks.com/aws/en/mlflow3/genai/)

In [0]:
# ── Cleanup & experiment URL ────────────────────────────────
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
experiment = mlflow.get_experiment_by_name(f"{notebook_path} Experiment")
if experiment:
    workspace_url = mlflow.utils.databricks_utils.get_workspace_url()
    print(f"Experiment URL: {workspace_url}/#mlflow/experiments/{experiment.experiment_id}")
    print(f"Experiment ID: {experiment.experiment_id}")
    print(f"\nTotal runs in this experiment can be viewed in the MLflow UI.")
else:
    print("Experiment not found. It will be created when you run the configuration cell.")